In [62]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [63]:
df = sns.load_dataset('tips')
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [64]:
# predict what is the time? if any one is visiting--is it lunch or dinner? time is the target variable

df.time.unique()

['Dinner', 'Lunch']
Categories (2, object): ['Lunch', 'Dinner']

In [65]:
# EDA >> Subjective

# Encoding, Missing value treatment >> automate

In [66]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
dtypes: category(4), float64(2), int64(1)
memory usage: 7.4 KB


In [67]:
df.isnull().sum()   # Still we will see null value treatment

,0
total_bill,0
tip,0
sex,0
smoker,0
day,0
time,0
size,0


In [68]:
# Since time is nominal variable, then we will use label encoder

from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df['time'] = encoder.fit_transform(df['time'])
df

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,0,2
1,10.34,1.66,Male,No,Sun,0,3
2,21.01,3.50,Male,No,Sun,0,3
3,23.68,3.31,Male,No,Sun,0,2
4,24.59,3.61,Female,No,Sun,0,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,0,3
240,27.18,2.00,Female,Yes,Sat,0,2
241,22.67,2.00,Male,Yes,Sat,0,2
242,17.82,1.75,Male,No,Sat,0,2


In [69]:
df['time'].unique()

array([0, 1])

In [70]:
X = df.drop('time', axis = 1)
y = df['time']

In [71]:
X.shape, y.shape

((244, 6), (244,))

In [72]:
X

,total_bill,tip,sex,smoker,day,size
0,16.99,1.01,Female,No,Sun,2
1,10.34,1.66,Male,No,Sun,3
2,21.01,3.50,Male,No,Sun,3
3,23.68,3.31,Male,No,Sun,2
4,24.59,3.61,Female,No,Sun,4
...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,3
240,27.18,2.00,Female,Yes,Sat,2
241,22.67,2.00,Male,Yes,Sat,2
242,17.82,1.75,Male,No,Sat,2


In [73]:
y

,time
0,0
1,0
2,0
3,0
4,0
...,...
239,0
240,0
241,0
242,0


In [74]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state=1)

In [75]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((195, 6), (49, 6), (195,), (49,))

In [76]:
X_train.head()

,total_bill,tip,sex,smoker,day,size
0,16.99,1.01,Female,No,Sun,2
154,19.77,2.00,Male,No,Sun,4
167,31.71,4.50,Male,No,Sun,4
110,14.00,3.00,Male,No,Sat,2
225,16.27,2.50,Female,Yes,Fri,2


In [77]:
X_test.head()

,total_bill,tip,sex,smoker,day,size
67,3.07,1.00,Female,Yes,Sat,1
243,18.78,3.00,Female,No,Thur,2
206,26.59,3.41,Male,Yes,Sat,3
122,14.26,2.50,Male,No,Thur,2
89,21.16,3.00,Male,No,Thur,2


In [78]:
y_train.head()

,time
0,0
154,0
167,0
110,0
225,1


In [79]:
y_test.head()

,time
67,0
243,0
206,0
122,1
89,1


In [80]:
# Handling the missing values
# Data Encoding
# Feature scaling

from sklearn.impute import SimpleImputer  # for missing value
from sklearn.preprocessing import OneHotEncoder  # for encoding
from sklearn.preprocessing import StandardScaler  # for scaling

from sklearn.pipeline import Pipeline  # a sequence of data transformer
from sklearn.compose import ColumnTransformer  # Groups all the pipeline steps for each of the columns

In [81]:
df.sample(1)

,total_bill,tip,sex,smoker,day,time,size
24,19.82,3.18,Male,No,Sat,0,2


In [82]:
cat_cols = ["sex", "smoker", "day"]
num_cols = ["total_bill", "tip", "size"]

In [89]:
# feature engineering automation using pipeline and column transformer

cat_pipeline = Pipeline(steps= [('imputation', SimpleImputer(strategy= 'most_frequent')),
                 ('scaling', OneHotEncoder())])

num_pipeline = Pipeline(steps= [('imputation', SimpleImputer(strategy= 'mean')),
                            ('encoding', StandardScaler())
                            ])

In [90]:
preprocessor = ColumnTransformer([("num_pipeline", num_pipeline, num_cols),
                   ("cat_pipeline", cat_pipeline, cat_cols)
                   ])
preprocessor

ColumnTransformer(transformers=[('num_pipeline',
                                 Pipeline(steps=[('imputation',
                                                  SimpleImputer()),
                                                 ('encoding',
                                                  StandardScaler())]),
                                 ['total_bill', 'tip', 'size']),
                                ('cat_pipeline',
                                 Pipeline(steps=[('imputation',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('scaling', OneHotEncoder())]),
                                 ['sex', 'smoker', 'day'])])

In [91]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [92]:
X_train

array([[-0.28611937, -1.47443803, -0.57766863, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.02695905, -0.71612531,  1.47042924, ...,  0.        ,
         1.        ,  0.        ],
       [ 1.3716196 ,  1.19880579,  1.47042924, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-0.23206267,  0.43283335, -0.57766863, ...,  0.        ,
         0.        ,  1.        ],
       [-1.06543688, -1.29060464, -0.57766863, ...,  1.        ,
         0.        ,  0.        ],
       [-0.29287646,  0.1034652 ,  0.44638031, ...,  1.        ,
         0.        ,  0.        ]])

In [93]:
X_test

array([[-1.85376383, -1.48209775, -1.60171757,  1.        ,  0.        ,
         0.        ,  1.        ,  0.        ,  1.        ,  0.        ,
         0.        ],
       [-0.08453291,  0.04984713, -0.57766863,  1.        ,  0.        ,
         1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         1.        ],
       [ 0.79501474,  0.36389583,  0.44638031,  0.        ,  1.        ,
         0.        ,  1.        ,  0.        ,  1.        ,  0.        ,
         0.        ],
       [-0.59356688, -0.33313909, -0.57766863,  0.        ,  1.        ,
         1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         1.        ],
       [ 0.18349826,  0.04984713, -0.57766863,  0.        ,  1.        ,
         1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         1.        ],
       [-1.32783714, -1.14506988, -0.57766863,  0.        ,  1.        ,
         0.        ,  1.        ,  0.        ,  1.        ,  0.        ,
         0.   

In [94]:
#now data is ready, lets build the multiple models

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

In [96]:
models = {
    "support vector classifier": SVC(),
    "DT Classifier": DecisionTreeClassifier(),
    "logistic regression": LogisticRegression()
}
models

{'support vector classifier': SVC(),
 'DT Classifier': DecisionTreeClassifier(),
 'logistic regression': LogisticRegression()}

In [99]:
for i in range(len(models)):
  print(list(models.values())[i])

SVC()
DecisionTreeClassifier()
LogisticRegression()


In [100]:
from sklearn.metrics import accuracy_score

def model_train_evaluation(X_train, X_test, y_train, y_test, models):
  evaluation = {}
  for i in range(len(models)):
    model = list(models.values())[i]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    model_score = accuracy_score(y_test, y_pred)
    evaluation[list(models.keys())[i]] = model_score

  return evaluation

In [101]:
model_train_evaluation(X_train, X_test, y_train, y_test, models)

{'support vector classifier': 0.9183673469387755,
 'DT Classifier': 0.9183673469387755,
 'logistic regression': 0.9183673469387755}

In [107]:
from sklearn.naive_bayes import GaussianNB

In [108]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision tree": DecisionTreeClassifier(),
    "Support vector classifier": SVC(),
    "Naive Bayes": GaussianNB()
}
models

{'Logistic Regression': LogisticRegression(),
 'Decision tree': DecisionTreeClassifier(),
 'Support vector classifier': SVC(),
 'Naive Bayes': GaussianNB()}

In [109]:
model_train_evaluation(X_train, X_test, y_train, y_test, models)

{'Logistic Regression': 0.9183673469387755,
 'Decision tree': 0.9387755102040817,
 'Support vector classifier': 0.9183673469387755,
 'Naive Bayes': 0.9183673469387755}